Step 1. Data Collection & Preprocessing 

Step 1A: Data Collection

In [1]:
# !pip install pymatgen matminer pandas scikit-learn numpy

In [1]:
import pandas as pd
import numpy as np

from mp_api.client import MPRester
from pymatgen.core import Composition

from matminer.featurizers.composition import (
    ElementProperty,
    Stoichiometry,
    IonProperty
)

from sklearn.preprocessing import StandardScaler



In [2]:
# Export MP API key
!pip install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv()

MP_API_KEY = os.getenv("MP_API_KEY")

if MP_API_KEY is None:
    raise RuntimeError("MP_API_KEY not found. Check your .env file.")

print("MP API key loaded successfully")


MP API key loaded successfully


In [5]:
# Query data from MP
with MPRester(MP_API_KEY) as mpr:
    results = mpr.materials.summary.search(
        # formula="*O6",
        elements=["O"],
        num_elements=4,
        energy_above_hull=(0, 0.10),
        fields=[
            "material_id",
            "formula_pretty",
            "structure",
            "band_gap",
            "energy_above_hull",
            "formation_energy_per_atom"
        ],
        chunk_size=1000
    )

records = [r.dict() for r in results]

df_raw = pd.DataFrame(records)
df_raw.head()

Retrieving SummaryDoc documents:   0%|          | 0/22993 [00:00<?, ?it/s]

,formula_pretty,material_id,structure,formation_energy_per_atom,energy_above_hull,band_gap,fields_not_requested
0,LiV3OF11,mp-coymc,"{'@module': 'pymatgen.core.structure', '@class...",-2.956297,0.054355,0.0007,"[builder_meta, nsites, elements, nelements, co..."
1,LiV4OF11,mp-coymj,"{'@module': 'pymatgen.core.structure', '@class...",-3.114582,0.048778,0.8691,"[builder_meta, nsites, elements, nelements, co..."
2,LiV3OF11,mp-coyml,"{'@module': 'pymatgen.core.structure', '@class...",-2.941772,0.068880,0.0000,"[builder_meta, nsites, elements, nelements, co..."
3,Li4V3OF11,mp-cozkt,"{'@module': 'pymatgen.core.structure', '@class...",-3.083630,0.081179,0.0000,"[builder_meta, nsites, elements, nelements, co..."
4,Li4V3OF11,mp-cozok,"{'@module': 'pymatgen.core.structure', '@class...",-3.089093,0.075716,0.2159,"[builder_meta, nsites, elements, nelements, co..."


In [6]:
print(df_raw.columns.tolist())
# Expected Output: ['formula_pretty', 'material_id', 'structure', 'energy_above_hull', 'band_gap', 'fields_not_requested']

['formula_pretty', 'material_id', 'structure', 'formation_energy_per_atom', 'energy_above_hull', 'band_gap', 'fields_not_requested']


In [7]:
from pymatgen.core import Composition
import numpy as np

A_SITE_ELEMENTS = {
    "Li", "Na", "K", "Rb", "Cs",
    "Mg", "Ca", "Sr", "Ba",
    "La", "Ce", "Pr", "Nd", "Sm", "Gd", "Y"
}

NETWORK_FORMERS = {
    "Si", "Ge", "P", "S", "B", "As", "Se", "Te"
}

def is_double_perovskite_relaxed(formula):
    try:
        comp = Composition(formula)
        el_amt = comp.get_el_amt_dict()

        # Must contain oxygen
        if "O" not in el_amt:
            return False

        # Must have exactly 4 elements: A, B, B', O
        if len(el_amt) != 4:
            return False

        o_amt = el_amt["O"]
        if o_amt <= 0:
            return False

        # Normalize all atom counts so oxygen = 6
        scale = 6 / o_amt
        norm = {el: amt * scale for el, amt in el_amt.items()}

        # Oxygen should normalize to 6
        if not np.isclose(norm["O"], 6, atol=1e-3):
            return False

        cations = {el: amt for el, amt in norm.items() if el != "O"}

        # Expected normalized cation counts: A2 B1 B'1
        counts = sorted(cations.values())
        if not all(np.isclose(a, b, atol=1e-2) for a, b in zip(counts, [1, 1, 2])):
            return False

        # Identify A-site element
        A_candidates = [el for el, amt in cations.items() if np.isclose(amt, 2, atol=1e-2)]
        if len(A_candidates) != 1:
            return False

        A = A_candidates[0]

        if A not in A_SITE_ELEMENTS:
            return False

        # Remove common oxyanion/network-forming chemistries
        for el in cations:
            if el in NETWORK_FORMERS:
                return False

        return True

    except Exception:
        return False

In [8]:
df_dp = df_raw[df_raw["formula_pretty"].apply(is_double_perovskite_relaxed)]
df_dp = df_dp.reset_index(drop=True)

print("Double perovskites found:", len(df_dp))
df_dp[["material_id", "formula_pretty", "band_gap"]].head()

Double perovskites found: 1080


,material_id,formula_pretty,band_gap
0,mp-chpaz,Mg2VWO6,1.7250
1,mp-chpcr,Mg2AgWO6,0.0556
2,mp-chpdg,Ca2AgWO6,0.0644
3,mp-chpnt,Mg2CuWO6,0.0000
4,mp-cotqi,Sr2FeMoO6,0.6149


Step 1B: Target Cleaning
Remove missing band gaps
Remove metallic entries (Eg = 0)

In [9]:
print(df_dp.columns)

Index(['formula_pretty', 'material_id', 'structure',
       'formation_energy_per_atom', 'energy_above_hull', 'band_gap',
       'fields_not_requested'],
      dtype='object')


In [10]:
df_ml_all = df_dp.copy()

# Remove rows with missing required target values
required_cols = [
    "band_gap",
    "energy_above_hull",
    "formation_energy_per_atom"
]

df_ml_all = df_ml_all.dropna(subset=required_cols).reset_index(drop=True)

# Dataset including metals
df_ml_all["is_metal"] = df_ml_all["band_gap"] <= 0

# Semiconductor-only dataset for band gap regression
df_ml_semiconductor = df_ml_all[df_ml_all["band_gap"] > 0].copy().reset_index(drop=True)

print("All valid double perovskites:", len(df_ml_all))
print("Semiconducting double perovskites:", len(df_ml_semiconductor))

df_ml_semiconductor[
    [
        "material_id",
        "formula_pretty",
        "band_gap",
        "energy_above_hull",
        "formation_energy_per_atom"
    ]
].head()

All valid double perovskites: 1080
Semiconducting double perovskites: 676


,material_id,formula_pretty,band_gap,energy_above_hull,formation_energy_per_atom
0,mp-chpaz,Mg2VWO6,1.7250,0.021626,-2.610230
1,mp-chpcr,Mg2AgWO6,0.0556,0.087788,-2.145787
2,mp-chpdg,Ca2AgWO6,0.0644,0.085028,-2.361840
3,mp-cotqi,Sr2FeMoO6,0.6149,0.000000,-2.532586
4,mp-cukwy,Sr2MnMoO6,0.8079,0.035374,-2.613435


STEP 1C — Feature engineering
Convert formula → composition
Generate Magpie descriptors
Save feature matrix

In [11]:
df_ml_semiconductor["composition"] = df_ml_semiconductor["formula_pretty"].apply(Composition)

In [12]:
from pymatgen.core import Composition
from matminer.featurizers.composition import (
    ElementProperty,
    Stoichiometry,
    IonProperty
)

# Use semiconductor dataset for band gap model
df_feat = df_ml_semiconductor.copy()
df_feat["composition"] = df_feat["formula_pretty"].apply(Composition)

# Magpie features
ep = ElementProperty.from_preset("magpie", impute_nan=True)
df_feat = ep.featurize_dataframe(
    df_feat,
    col_id="composition",
    ignore_errors=True
)

# Stoichiometric features
stoich = Stoichiometry()
df_feat = stoich.featurize_dataframe(
    df_feat,
    col_id="composition",
    ignore_errors=True
)

# Ionic-property features
ion = IonProperty(fast=True, impute_nan=True)
df_feat = ion.featurize_dataframe(
    df_feat,
    col_id="composition",
    ignore_errors=True
)

print("Feature dataframe shape:", df_feat.shape)
print("NaNs in dataframe:", df_feat.isna().sum().sum())

ElementProperty:   0%|          | 0/676 [00:00<?, ?it/s]

Stoichiometry:   0%|          | 0/676 [00:00<?, ?it/s]

IonProperty:   0%|          | 0/676 [00:00<?, ?it/s]

Feature dataframe shape: (676, 150)
NaNs in dataframe: 0


In [13]:
non_feature_cols = [
    "material_id",
    "formula_pretty",
    "composition",
    "structure",
    "band_gap",
    "energy_above_hull",
    "formation_energy_per_atom",
    "is_stable",
    "theoretical",
    "is_metal",
    "composition_reduced"
]

X_ml = df_feat.drop(
    columns=[c for c in non_feature_cols if c in df_feat.columns],
    errors="ignore"
)

# Keep only numeric columns
X_ml = X_ml.select_dtypes(include=[np.number])

# Replace inf values if any
X_ml = X_ml.replace([np.inf, -np.inf], np.nan)

# Drop columns that are entirely NaN
X_ml = X_ml.dropna(axis=1, how="all")

# Fill remaining NaNs with column medians
X_ml = X_ml.fillna(X_ml.median())

y_gap = df_feat["band_gap"]
y_hull = df_feat["energy_above_hull"]
y_form = df_feat["formation_energy_per_atom"]

print("Final feature matrix shape:", X_ml.shape)
print("Band gap target shape:", y_gap.shape)
print("Ehull target shape:", y_hull.shape)
print("Formation energy target shape:", y_form.shape)
print("Remaining NaNs:", X_ml.isna().sum().sum())

Final feature matrix shape: (676, 140)
Band gap target shape: (676,)
Ehull target shape: (676,)
Formation energy target shape: (676,)
Remaining NaNs: 0


In [14]:
import joblib

# Save full cleaned datasets
df_ml_all.to_csv("double_perovskite_all_valid.csv", index=False)
df_ml_semiconductor.to_csv("double_perovskite_semiconductor.csv", index=False)

# Save feature dataframe and ML matrices
df_feat.to_csv("double_perovskite_featurized.csv", index=False)

joblib.dump(X_ml, "dp_features_magpie_stoich_ion.pkl")
joblib.dump(y_gap, "dp_bandgap_targets.pkl")
joblib.dump(y_hull, "dp_ehull_targets.pkl")
joblib.dump(y_form, "dp_formation_energy_targets.pkl")

print("Updated data collection and feature engineering completed.")

Updated data collection and feature engineering completed.
